# Getting started

This tutorial builds a specification, draws graphs from it with different seeds and
specifications, compares them, and saves one. It runs in a few seconds.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import bipgen

## A specification

A dictionary with the layout of the YAML format: 2000 BCN and 3000 TGT in the unit
disk, lognormal target degrees, a Gaussian kernel of scale 0.12 and 5% background
edges.

In [ ]:
base = {
    "sizes": {"n_bcn": 2000, "n_tgt": 3000},
    "degrees": {"distribution": "lognormal",
                "bcn": {"median": 15, "sigma": 0.5},
                "tgt": {"sigma": 0.6}},  # shape only: scale set by the BCN total
    "edges": {"model": "kernel", "kernel": "gaussian", "scale": 0.12},
    "background": {"model": "uniform", "ratio": 0.05},
}
spec = bipgen.spec_from_dict(base)
spec.to_dict()

## One realization

The seed comes from the specification (42 by default) unless it is given in the call.
`info` records the seed, the target number of edges, the expected number after
clipping probabilities at 1, and the convergence of Sinkhorn scaling.

In [ ]:
graph = bipgen.generate(spec)
print(graph)
graph.info

In [ ]:
graph.edges().head()

## Seeds

The same seed gives the same graph; other seeds give other realizations of the same
ensemble.

In [ ]:
same = bipgen.generate(spec, seed=42)
print("same seed, edges that differ:", (graph.biadjacency != same.biadjacency).nnz)

ensemble = [bipgen.generate(spec, seed=s) for s in range(5)]
print("edges in five realizations:", [g.n_edges for g in ensemble])

## Different specifications

The same sizes and degrees with a short kernel, a long one, and the long one with
more background edges. Sinkhorn scaling matches the target degrees before
probabilities are clipped at 1: with the short kernel, clipping removes part of the
edges, and `expected_edges` measures it.

In [ ]:
variants = {
    "scale 0.10": {**base, "edges": {"model": "kernel", "scale": 0.10}},
    "scale 0.25": {**base, "edges": {"model": "kernel", "scale": 0.25}},
    "scale 0.25, 20% background": {**base, "edges": {"model": "kernel", "scale": 0.25},
                                   "background": {"model": "uniform", "ratio": 0.2}},
}
graphs = {name: bipgen.generate(bipgen.spec_from_dict(v)) for name, v in variants.items()}
for name, g in graphs.items():
    print(f"{name:28s} {g}  expected / target edges = "
          f"{g.info['expected_edges'] / g.info['target_edges']:.3f}")

## Comparing graphs

One row per graph: nodes and a subset of edges, edge lengths, and the degree
distributions of the two sides, realized against target. A longer kernel gives
longer edges; background edges have lengths unrelated to distance and add to the
degrees.

In [ ]:
fig, axes = plt.subplots(len(graphs), 4, figsize=(15, 3.6 * len(graphs)))
for row, (name, g) in zip(axes, graphs.items()):
    bipgen.plot_graph(g, ax=row[0], max_edges=2000, node_size=1)
    bipgen.plot_edge_lengths(g, ax=row[1])
    bipgen.plot_degrees(g, axes=row[2:])
    row[0].set_title(f"{name}\n{row[0].get_title()}", fontsize=9)
fig.tight_layout()

The same summary statistics on each graph. To test an observed graph against an
ensemble, see [Presets and comparison](02_presets_and_comparison.ipynb).

In [ ]:
pd.DataFrame({name: bipgen.statistics(g) for name, g in graphs.items()}).round(2)

## k nearest neighbours

Each BCN links to its `k` nearest TGT; no degree section is needed.

In [ ]:
knn = bipgen.spec_from_dict({"sizes": {"n_bcn": 1500, "n_tgt": 1500},
                             "edges": {"model": "knn", "k": 6}})
g = bipgen.generate(knn)
bipgen.plot_degrees(g);

## Spatially correlated density

Cell counts with coefficient of variation 0.8, arranged to bring the spatial correlation
between nearby cells (Moran's I) at lags 1 and 2 towards the target. The correlogram
reached is in `info`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, rho in zip(axes, ([0.1, 0.05], [0.7, 0.5])):
    s = bipgen.spec_from_dict({**base, "positions": {"method": "metropolis", "density": {
        "occupancy": {"source": "gamma", "cv": 0.8}, "correlogram": rho, "sweeps": 200}}})
    g = bipgen.generate(s)
    bipgen.plot_graph(g, ax=ax, max_edges=0, node_size=2)
    ax.set_title(f"target {rho}, reached {np.round(g.info['correlogram'], 2).tolist()}",
                 fontsize=9)

## 3D

The domain becomes the unit ball.

In [ ]:
g3 = bipgen.generate(bipgen.spec_from_dict(
    {**base, "dim": 3, "edges": {"model": "kernel", "scale": 0.2}}))
print(g3)
fig = plt.figure(figsize=(5.5, 5.5))
bipgen.plot_graph(g3, ax=fig.add_subplot(projection="3d"), max_edges=0, node_size=2);

Spatially correlated densities work in 3D too, on a grid of cubic cells clipped to the
unit ball. A thin slice through the centre shows the clusters better than the whole
cloud.

In [ ]:
from bipgen.plot import COLORS

s3 = bipgen.spec_from_dict({**base, "dim": 3, "edges": {"model": "kernel", "scale": 0.2},
                            "positions": {"method": "metropolis", "density": {
                                "correlogram": [0.7, 0.5], "sweeps": 200}}})
g3 = bipgen.generate(s3)
pts = np.vstack([g3.pos_bcn, g3.pos_tgt])
slab = pts[np.abs(pts[:, 2]) < 0.1]
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(slab[:, 0], slab[:, 1], s=3, color=COLORS["bcn"], linewidths=0)
ax.set_aspect("equal")
ax.set_title(f"thin slice |z| < 0.1 of a 3D correlated density, {len(slab)} nodes", fontsize=9);

## Saving

One directory per graph: edge table, node table, specification and diagnostics.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = bipgen.save(graph, Path(tmp) / "graph_042")
    print(sorted(p.name for p in path.iterdir()))
    again = bipgen.load(path)
print(again)
print("edges that differ after reloading:", (again.biadjacency != graph.biadjacency).nnz)